In [0]:
import os
import json
import random
from datetime import datetime, timedelta, timezone

dbutils.widgets.text("batch_id", "1", "Batch Number (1, 2, 3...)")
batch_id = int(dbutils.widgets.get("batch_id"))
random.seed(100 + batch_id)

base_volume_path = "/Volumes/fintech_prod/raw_landing/cdc_landing_vol"
wallets_dir = f"{base_volume_path}/wallets_cdc"
transactions_dir = f"{base_volume_path}/transactions_cdc"

os.makedirs(wallets_dir, exist_ok=True)
os.makedirs(transactions_dir, exist_ok=True)

base_time = datetime(2026, 11, 1, 8, 0, 0, tzinfo=timezone.utc) + timedelta(hours=batch_id)
wallet_events = []

for i in range(1, 11):
    wallet_id = f"W100{i}" if i < 10 else f"W10{i}"
    is_update = batch_id > 1 and i <= 4
    event = {
        "wallet_id": wallet_id,
        "customer_id": f"CUST-{500 + i}",
        "kyc_tier": "TIER_2" if is_update else "TIER_1",
        "risk_status": random.choice(["MEDIUM", "HIGH"]) if is_update else "LOW",
        "country_code": "KE",
        "cdc_operation": "UPDATE" if is_update else "INSERT",
        "event_timestamp": (base_time + timedelta(minutes=i)).isoformat()
    }
    if batch_id == 1 or is_update:
        wallet_events.append(event)

wallet_file_path = f"{wallets_dir}/wallets_batch_{batch_id:02d}.json"
with open(wallet_file_path, "w") as f:
    for record in wallet_events:
        f.write(json.dumps(record) + "\n")

print(f"Batch {batch_id}: Landed {len(wallet_events)} wallet CDC events to {wallet_file_path}")

In [0]:
merchants = ["M-SAFARI-01", "M-KCB-02", "M-EQUITY-03", "M-AIRTEL-04"]
tx_events = []

for t in range(1, 21):
    tx_id = f"TX-{batch_id * 100 + t}"
    is_poison = (t == 20)
    status = "INITIATED" if batch_id == 1 else random.choice(["AUTHORIZED", "SETTLED", "REVERSED"])
    tx_record = {
        "transaction_id": tx_id,
        "wallet_id": None if is_poison else f"W100{(t % 9) + 1}",
        "merchant_id": random.choice(merchants),
        "amount": round(random.uniform(-500.0, -10.0), 2) if is_poison else round(random.uniform(50.0, 5000.0), 2),
        "currency": "KES",
        "payment_status": status,
        "event_timestamp": (base_time + timedelta(minutes=t)).isoformat()
    }
    if batch_id > 1 and t <= 5:
        tx_record["transaction_id"] = f"TX-{100 + t}"
        tx_record["payment_status"] = "SETTLED" if t < 5 else "REVERSED"
    if batch_id > 1 and t == 6:
        tx_record["transaction_id"] = "TX-106"
        tx_record["event_timestamp"] = (base_time - timedelta(hours=2)).isoformat()
    if batch_id > 1:
        tx_record["gateway_latency_ms"] = random.randint(15, 250)
    tx_events.append(tx_record)

tx_file_path = f"{transactions_dir}/transactions_batch_{batch_id:02d}.json"
with open(tx_file_path, "w") as f:
    for record in tx_events:
        f.write(json.dumps(record) + "\n")

print(f"Batch {batch_id}: Landed {len(tx_events)} transaction CDC events to {tx_file_path}")